# LangGraph
Human-in-the-loop（人工干预介入）、State Checkpointing（状态快照恢复与断点续传）。
在金融转账、医疗诊断、数据库删除、邮件群发等高风险工程场景中，我们绝不能让 Agent 完全“无监管地自动运行”。我们需要一种机制：当 Agent 准备执行敏感高风险操作时，程序自动挂起（Pause），将当前完整状态（State）序列化保存；等到人类审查员在前端点击“批准”或“修改参数”后，系统再从断点处精准恢复（Resume）并继续执行。

## LangGraph 人工干预与状态持久化
1. Checkpointer 持久化机制：理解 `thread_id` 线程隔离原理，学习如何将 `Graph` 的每一步运行状态保存至内存（`MemorySaver`）或数据库（`SqliteSaver` / `PostgresSaver`）。
2. 实现 `Human-in-the-Loop` 挂起与恢复：学会利用 `interrupt_before` 和 `interrupt_after` 在高风险节点（如敏感 Tool 执行前）中断程序，等待人工审核。
3. 实现状态修改与“时间旅行”（State Editing & Time Travel）：学习人类干预员如何在程序挂起时修改状态中的参数（例如：将转账金额 10000 降为 1000），或回滚到任意历史节点重新跑分支。

#### Checkpointer 与 Human-in-the-Loop 架构
1. State Checkpointer（状态检查点/快照）

    传统 Chain 执行完即销毁上下文；而 LangGraph 引入了 Checkpointer（状态快照器）：
   * 工作机制：在每个 Node 执行完成后，Checkpointer 都会自动捕捉当前的 `AgentState` 并将其序列化持久化。
   * `thread_id` 线程隔离：每一个独立的对话或任务流水线都被赋予一个唯一的 `thread_id`。不同用户或不同任务的状态互不干扰。
   * 断点恢复能力：只要传入相同的 `configurable: {"thread_id": "xxx"}`，图引擎可以在服务器重启或长时间暂停后，瞬间重构出挂起时的完全一致的状态 `context`。

2. Human-in-the-Loop（人机协同三部曲）
    * 审批授权（Approval）：Agent 提出要调用 `execute_bank_transfer(amount=50000)`，Graph 在进入 `action` 节点前触发中断，人类审查后回复 `Approve` 授权执行，或回复 `Reject` 阻断执行。
    * 状态修正（State Editing）：人类审查员发现 Agent 计算出的参数有误，直接在挂起状态中手动修改 `state["messages"]` 或中间变量，修改后再让 Agent 继续往下走.
    * 回滚重试（Time Travel）：查看 `Checkpointer` 记录的所有历史 `Snapshot`，将图的状态强制回退到 3 步之前的某个节点重新探索不同分支。


下面的完整代码展示了如何使用 `MemorySaver` 作为 Checkpointer，结合 `interrupt_before` 构建一个在触发敏感支付工具前挂起并等待人类命令的 LangGraph 智能体：

In [ ]:
from typing import Annotated, TypedDict, Sequence, Literal
import operator

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# --- 1. 定义敏感操作工具 ---
@tool
def bank_transfer(to_account: str, amount: float) -> str:
    """高风险工具：执行银行转账支付操作"""
    return f"SUCCESS: 已成功向账号 {to_account} 转账 {amount} 元。"

tools = [bank_transfer]
tool_map = {t.name: t for t in tools}

# --- 2. 定义状态定义 (AgentState) ---
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# --- 3. 定义图节点 ---

def agent_node(state: AgentState):
    """LLM 决策节点 (模拟生成带参数的 Tool Call)"""
    messages = state["messages"]
    last_msg = messages[-1]

    # 如果最后一条消息是 ToolMessage，说明工具已执行完，生成总结回复
    if isinstance(last_msg, ToolMessage):
        return {"messages": [AIMessage(content=f"转账审批流程已完成！收到执行反馈：{last_msg.content}")]}

    # 模拟 LLM 解析用户意图后，决定调用高风险工具 bank_transfer
    ai_msg = AIMessage(
        content="",
        tool_calls=[{
            "name": "bank_transfer",
            "args": {"to_account": "622200123456", "amount": 8888.0},
            "id": "call_transfer_999"
        }]
    )
    return {"messages": [ai_msg]}

def action_node(state: AgentState):
    """高风险工具执行节点"""
    messages = state["messages"]
    last_msg = messages[-1]

    tool_outputs = []
    for tool_call in last_msg.tool_calls:
        tool_obj = tool_map[tool_call["name"]]
        output = tool_obj.invoke(tool_call["args"])
        tool_outputs.append(
            ToolMessage(content=str(output), tool_call_id=tool_call["id"])
        )
    return {"messages": tool_outputs}

def should_continue(state: AgentState) -> Literal["action", "end"]:
    messages = state["messages"]
    last_msg = messages[-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "action"
    return "end"

# --- 4. 构建包含 Checkpointer 与 Interrupt 的 StateGraph ---
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("action", action_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"action": "action", "end": END}
)
workflow.add_edge("action", "agent")

# 初始化内存检查点器
memory_checkpointer = MemorySaver()

# ⚠️ 关键点：设置 interrupt_before=["action"]
# 表示在进入 "action" 节点执行前，强行中断程序并将快照写入 checkpointer
app = workflow.compile(
    checkpointer=memory_checkpointer,
    interrupt_before=["action"]
)

# --- 5. 人工干预与审批流程测试 ---
if __name__ == "__main__":
    # 配置唯一的线程 ID，用于标识该笔交易对话
    thread_config = {"configurable": {"thread_id": "tx_session_20260730"}}

    print("="*60)
    print("🚀 第一阶段：用户发起转账请求...")
    print("="*60)

    initial_input = {"messages": [HumanMessage(content="帮我给账号 622200123456 转账 8888 元")]}

    # 第一次 stream 执行：Agent 会生成 Tool Call，但在进入 action 之前被中断
    for event in app.stream(initial_input, thread_config):
        print(f"📍 [节点执行]: {event}")

    # 获取此时的线程快照状态
    current_state = app.get_state(thread_config)
    print("\n⚠️ 【系统安全中断】检测到准备执行高风险节点 [action]！")
    print(f"⏸️ 当前图暂停位置 (Next Node): {current_state.next}")

    # 查看 Agent 拟调用的参数
    pending_tool_call = current_state.values["messages"][-1].tool_calls[0]
    print(f"🔍 人类审核员请校验拟执行的工具: {pending_tool_call['name']}")
    print(f"🔍 拟执行参数: {pending_tool_call['args']}\n")

    # 模拟人工审核过程 (Human Decision)
    human_input = input("👉 请输入审批指令 (输入 'y' 批准执行, 输入 'n' 拒绝并终止): ")

    if human_input.lower() == 'y':
        print("\n="*60)
        print("✅ 人类审核通过！恢复图执行 (Resume)...")
        print("="*60)
        # 传入 None 表示继续从挂起点恢复运行
        for event in app.stream(None, thread_config):
            print(f"📍 [恢复后节点执行]: {event}")

        final_state = app.get_state(thread_config)
        print(f"\n🎉 最终回答: {final_state.values['messages'][-1].content}")
    else:
        print("\n❌ 审核员拒绝该高风险操作，流程已安全拦截。")


1. State Editing 参数修改闭环：
    * 在上面的实战代码中，如果人类审核员认为“转账 8888 元太多，只批准转 1000 元”，我们应该如何使用 `app.update_state(thread_config, ...)` 在不重新开始对话的前提下，将` tool_calls` 中的 `amount` 改为 `1000.0` 之后再 `stream(None, thread_config)` 恢复执行？
    * 工程思考：修改现有消息的 `tool_calls` 参数与直接注入一条提示消息相比，哪种方式对保留 Agent 原始推理链更稳健？

2. 生产环境持久化存储选型：
    * 本代码中使用的是内存级的 `MemorySaver`，在 Python 进程重启后状态会丢失。
    * 思考：在真正的企业级多实例微服务架构中，应该选择 `SqliteSaver`、`PostgresSaver` 还是基于 `Redis` 的 `Custom Checkpointer`？
        * SqliteSaver: 多实例共享不支持，并发低，审计有限
        * PostgresSaver： 支持多实例共享，并发及写入能力强。适用于常规企业级多实例部署
        * RedisSaver： 支持多实例共享，并发及写入能力极强(内存级)，配置运维复杂。适合超高并发对延迟极度敏感的场景，如在线客服、实时语音助手。
    * 在多节点（Multi-replica）部署时，`thread_id` 如何防止并发竞争（Race Condition）？

        | 并发防护       | 手段                         | 解决的问题                |
        |------------|----------------------------|----------------------|
        | 应用层串行化     | 分布式锁 / 消息队列                | 同一 `thread_id` 多请求竞争 |
        | 命名空间隔离     | `tenant:session:task` 三段式  | 不同会话/任务串号            |
        | 幂等性        | `thread_id + turn` 幂等键     | 重试导致 Tool 重复执行       |
        | Reducer 合并 | `Annotated[T, reducer_fn]` | 并行节点写入同一字段覆盖         |

